In [1]:
import os
import numpy as np
import cv2
from scipy.ndimage import gaussian_filter

In [2]:
# 创建保存路径
gt_path = "F:/B208/DeepLearning/thrid_generation/binary_test"
input_path = "F:/B208/DeepLearning/thrid_generation/gray_test"
os.makedirs(gt_path, exist_ok=True)
os.makedirs(input_path, exist_ok=True)

In [3]:
# 图像大小
image_size = 256

In [4]:
# 贝塞尔曲线函数
def bernstein_poly(t, n, i):
    """计算贝塞尔基函数"""
    return np.math.comb(n, i) * (t**i) * ((1 - t)**(n - i))

def bezier_curve(control_points, num_points=100):
    """生成贝塞尔曲线"""
    n = len(control_points) - 1  # 控制点数量
    t_values = np.linspace(0, 1, num_points)
    curve = np.zeros((num_points, 2))  # 初始化曲线

    for i in range(n + 1):
        curve += np.array([bernstein_poly(t, n, i) for t in t_values]).reshape(-1, 1) * control_points[i]

    return curve

# 随机生成贝塞尔曲线
def generate_random_bezier_curve(image_size):
    """在图像上绘制随机贝塞尔曲线"""
    control_points = np.random.randint(10, image_size - 10, size=(4, 2))  # 随机4个控制点
    curve = bezier_curve(control_points, num_points=200)

    curve_image = np.zeros((image_size, image_size), dtype=np.uint8)
    for i in range(len(curve) - 1):
        pt1 = tuple(curve[i].astype(int))
        pt2 = tuple(curve[i + 1].astype(int))
        cv2.line(curve_image, pt1, pt2, 255, 1)  # 绘制线段连接点
    return curve_image, control_points

# 生成随机曲线和少量直线
def generate_curves_and_lines(image_size):
    """生成随机曲线和少量直线"""
    img = np.zeros((image_size, image_size), dtype=np.uint8)
    gt_data = []
    used_columns = set()
    
    # 生成随机曲线
    for _ in range(np.random.randint(5, 21)):  # 大部分为曲线
        curve_img, control_points = generate_random_bezier_curve(image_size)
        img = cv2.add(img, curve_img)
        gt_data.append(("curve", control_points))  # 保存曲线控制点

    # 生成少部分直线
    num_lines = np.random.randint(10, 31)
    for _ in range(num_lines):
        max_attempts = 100  # 最大尝试次数
        attempts = 0
        while attempts < max_attempts:
            attempts += 1
            x1 = np.random.randint(0, image_size)  # 生成随机列
            # 检查是否与已使用列间隔至少 3 像素
            if all(abs(x1 - used_col) >= 3 for used_col in used_columns):
                y1 = np.random.randint(0, 50)  # 起始位置在图像顶部50像素内
                length = np.random.randint(50, 301)  # 随机线段长度
                x2, y2 = x1, min(y1 + length, image_size - 1)  # 计算终点
                gt_data.append(("line", ((x1, y1), (x2, y2))))  # 保存直线数据
                cv2.line(img, (x1, y1), (x2, y2), 255, 1)  # 绘制直线
                used_columns.add(x1)  # 记录已使用的列
                break
        else:
            print(f"Warning: Could not place a line after {max_attempts} attempts.")
            
    return img, gt_data

In [5]:
# 应用亮度调整和高斯模糊
def apply_brightness_and_blur(img, gt_data):
    blurred_img = np.zeros_like(img, dtype=np.float32)
    for item in gt_data:
        if item[0] == "line":  # 处理直线
            ((x1, y1), (x2, y2)) = item[1]
            brightness = np.random.randint(150, 256)
            temp_img = np.zeros_like(img, dtype=np.uint8)
            cv2.line(temp_img, (x1, y1), (x2, y2), brightness, 1)
            sigma = np.random.uniform(0.8, 2.4)
            temp_img = gaussian_filter(temp_img.astype(np.float32), sigma=sigma)
            blurred_img += temp_img
        elif item[0] == "curve":  # 处理曲线
            control_points = item[1]
            brightness = np.random.randint(150, 256)
            curve = bezier_curve(control_points, num_points=200)
            temp_img = np.zeros_like(img, dtype=np.uint8)
            for i in range(len(curve) - 1):
                pt1 = tuple(curve[i].astype(int))
                pt2 = tuple(curve[i + 1].astype(int))
                cv2.line(temp_img, pt1, pt2, brightness, 1)
            sigma = np.random.uniform(0.8, 3.6)
            temp_img = gaussian_filter(temp_img.astype(np.float32), sigma=sigma)
            blurred_img += temp_img
    return np.clip(blurred_img, 0, 255).astype(np.uint8)

In [6]:
# 添加背景和噪声
def add_background_and_noise(img):
    background_value = np.random.randint(30, 121)
    background = np.full_like(img, background_value, dtype=np.uint8)
    noisy_img = cv2.addWeighted(img, 1, background, 1, 0)

    # 添加噪声
    noise = np.random.normal(0, 10, img.shape).astype(np.float32)
    noisy_img = noisy_img.astype(np.float32) + noise
    return np.clip(noisy_img, 0, 255).astype(np.uint8)

In [ ]:
# 主程序
for i in range(200):  # 生成10张图像
    base_img, gt_data = generate_curves_and_lines(image_size)

    # 保存原始图像
    gt_save_path = os.path.join(gt_path, f"randomgt_{i:04d}.png")
    cv2.imwrite(gt_save_path, base_img)

    # 应用亮度调整和模糊
    blurred_img = apply_brightness_and_blur(base_img, gt_data)

    # 添加背景和噪声
    noisy_img = add_background_and_noise(blurred_img)

    # 保存含噪声图像
    input_save_path = os.path.join(input_path, f"randominput_{i:04d}.png")
    cv2.imwrite(input_save_path, noisy_img)